In [1]:
%cd /content
![ -d dl-hw4-sound-integration ] || git clone https://github.com/boldirev-as/dl-hw4-sound-integration.git
%cd /content/dl-hw4-sound-integration
%pip install -q -r requirements.txt

/content
Cloning into 'dl-hw4-sound-integration'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 93 (delta 40), reused 93 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 24.30 KiB | 428.00 KiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/dl-hw4-sound-integration
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.3 MB/s eta 0:00:00
   ━━━

In [2]:
%%bash
set -e
mkdir -p /content/data
if [ ! -d /content/data/LibriSpeech/test-clean ]; then
  wget -q --show-progress -O /content/test-clean.tar.gz https://www.openslr.org/resources/12/test-clean.tar.gz
  tar -xzf /content/test-clean.tar.gz -C /content/data
fi
find /content/data/LibriSpeech/test-clean -name '*.flac' | wc -l

2620



     0K .......... .......... .......... .......... ..........  0%  165K 34m9s
    50K .......... .......... .......... .......... ..........  0% 1.22M 19m20s
   100K .......... .......... .......... .......... ..........  0%  443K 17m8s
   150K .......... .......... .......... .......... ..........  0% 1.28M 13m55s
   200K .......... .......... .......... .......... ..........  0%  440K 13m42s
   250K .......... .......... .......... .......... ..........  0% 33.0M 11m27s
   300K .......... .......... .......... .......... ..........  0% 27.5M 9m50s
   350K .......... .......... .......... .......... ..........  0% 11.8M 8m40s
   400K .......... .......... .......... .......... ..........  0% 1.53M 8m6s
   450K .......... .......... .......... .......... ..........  0%  452K 8m32s
   500K .......... .......... .......... .......... ..........  0% 20.5M 7m47s
   550K .......... .......... .......... .......... ..........  0% 30.9M 7m9s
   600K .......... .......... .......... ........

In [3]:
from pathlib import Path
import yaml

with open("configs/config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["device"] = "cuda"
config["data"]["valid_root"] = "/content/data/LibriSpeech/test-clean"
config["data"]["eval_num_files"] = None
config["eval"]["checkpoint_path"] = "checkpoints/final.pt"
config["eval"]["output_path"] = "outputs/full_eval_metrics.json"
config["eval"]["use_nisqa"] = True
config["checkpoint_download"]["output_path"] = "checkpoints/final.pt"

Path("outputs").mkdir(exist_ok=True)
Path("checkpoints").mkdir(exist_ok=True)

with open("configs/config.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False)

config["data"]

{'train_root': '/kaggle/input/datasets/victorling/librispeech-clean/LibriSpeech/train-clean-100',
 'valid_root': '/content/data/LibriSpeech/test-clean',
 'eval_num_files': None}

In [4]:
!python3 download_checkpoints.py
!python3 evaluate.py

Downloading...
From (original): https://drive.google.com/uc?id=1cxuhJ-zLqiVbRreARDhfeih6f_39T3Rb
From (redirected): https://drive.google.com/uc?id=1cxuhJ-zLqiVbRreARDhfeih6f_39T3Rb&confirm=t&uuid=daba9fce-e2e7-4ee9-bd83-065df23efdab
To: /content/dl-hw4-sound-integration/checkpoints/final.pt
100% 293M/293M [00:05<00:00, 53.1MB/s]
Checkpoint saved to checkpoints/final.pt
Evaluating:   0% 0/2620 [00:00<?, ?it/s]downloading https://github.com/gabrielmittag/NISQA/raw/refs/heads/master/weights/nisqa.tar to /root/.torchmetrics/NISQA/nisqa.tar
Evaluating: 100% 2620/2620 [13:56<00:00,  3.13it/s]
{'nisqa': 2.423790242972265, 'stoi': 0.8372344430166346, 'mel': 0.6543399197227172}


In [5]:
import json

with open("outputs/full_eval_metrics.json", "r", encoding="utf-8") as file:
    result = json.load(file)

print(json.dumps(result["mean"], indent=2))

{
  "nisqa": 2.423790242972265,
  "stoi": 0.8372344430166346,
  "mel": 0.6543399197227172
}
